# Stage 1 — 10x glomerular segmentation

Resolve one session, build per-odor correlation maps, curate a shared mask, and extract traces.


In [ ]:
%load_ext autoreload
%autoreload 2

import os, shutil, sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

sys.path.insert(0, "..")

from analysis.session.devshim import LocalGroup
from analysis.session.resolve import resolve_group
from analysis.session.zscore import make_group_keys
from analysis.session.corrcache import build_group_correlation_maps
from analysis.seg_10x.watershed import GLOM_10X_DEFAULTS, scale_params

GROUP_ID = 198
PORTABLE_MASK_BUNDLE = None  # optional server HDF5 for extraction-only runs
MANIPULATION = ""
APPROVED_ONLY = False
EXCLUDE_ACQ = []
SEPARATE_BY_CONDITION = False
DETREND = True
PRE_S, POST_S, SIGMA_PX = 2.0, 3.0, 0.5

# Keep large temporary writes local; set these roots once per machine.
SCRATCH = Path(os.environ.get("ODYN_SCRATCH_ROOT", Path.home() / "odyn_scratch"))
MAIN = Path(os.environ.get("ODYN_IMAGING_ROOT", "/Volumes/MossLab/ImagingData"))
SCRATCH.mkdir(parents=True, exist_ok=True)
CORR_CACHE = SCRATCH / "correlation_cache" / f"group{GROUP_ID}"

# A local metadata copy avoids repeated slow reads from the server. The resolver
# uses frame sync when available and otherwise the best acquisition timing source.
group = LocalGroup(
    MAIN / ".odyn" / "odyn.db", MAIN,
    snapshot_to=SCRATCH / "odyn_snapshot.db", max_age_s=1800,
)
session = resolve_group(
    group, group_id=GROUP_ID, manipulation=MANIPULATION,
    approved_only=APPROVED_ONLY, exclude_acq_ids=tuple(EXCLUDE_ACQ),
)
print(session.summary())


## Correlation maps

Each trial is baseline-z-scored, averaged within odor group, and reduced to an 8-neighbour local-correlation map. Accumulators and correlation caches stay on local scratch to avoid server traffic.


In [ ]:
keys = make_group_keys(
    session.odor_ids, session.states,
    separate_by_condition=SEPARATE_BY_CONDITION,
)
WORK = SCRATCH / f"nb_work_{GROUP_ID}"

corr_by_odor, corr_meta = build_group_correlation_maps(
    session.paths,
    odor_on_frames=session.odor_on_frames,
    odor_off_frames=session.odor_off_frames,
    group_keys=keys,
    frame_rate=session.frame_rate,
    pre_s=PRE_S,
    post_s=POST_S,
    spatial_sigma_px=SIGMA_PX,
    work_dir=WORK,
    cache_dir=CORR_CACHE,
)
print(f"{len(corr_by_odor)} maps; cache {corr_meta['cache']}")


## Segmentation GUI

The GUI moves through three states:

1. **Tune** — adjust segmentation for all odors or override one odor. Threshold controls sensitivity; adaptive thresholding handles uneven backgrounds; diameter limits set ROI size; peak distance controls watershed splitting; border excludes edge artifacts.
2. **Merge** — freeze segmentation and combine matching detections. Minimum overlap controls matching, minimum detections requires support across odors, and consensus fraction controls how much of the overlapping footprint is retained.
3. **Curate** — freeze parameters, then add, delete, or exclude ROIs on the merged mask. Returning to an earlier state discards later edits.

Manual additions grow from watershed seeds. **Save masks + config** stores the curated mask and settings.


In [ ]:
from analysis.seg_10x.gui import launch

params = scale_params(GLOM_10X_DEFAULTS, to_um_per_px=session.um_per_px)
gui = launch(
    corr_by_odor,
    save_path=SCRATCH / f"masks_group{GROUP_ID}.npz",
    params=params,
)


## Final mask

Run after GUI curation. This writes a compact mask-only HDF5 to the session output directory. Set `PORTABLE_MASK_BUNDLE` to that file on another computer, then run the setup, final-mask, and extraction cells without rebuilding correlation maps.


In [ ]:
from analysis.session.finalize import mask_hash
from analysis.session.masks import (
    background_image, load_latest_mask, load_mask_bundle,
    save_mask_bundle, save_mask_overlay,
)
from analysis.session.store import session_filename

def output_path(kind, suffix):
    return session.output_dir / session_filename(
        group_id=session.group_id, exp_name=session.exp_name,
        kind=kind, suffix=suffix,
    )

portable = (load_mask_bundle(PORTABLE_MASK_BUNDLE)
            if PORTABLE_MASK_BUNDLE is not None else None)
saved = portable or load_latest_mask(session.output_dir)

if "gui" in dir() and gui.state.phase == "curate":
    labels = gui.state.curated_mask()
    masks_by_group = gui.state.segment_all()
    reference = background_image(corr_by_odor)
    params = dict(gui.state.shared)
    merge_params = dict(gui.state.merge_params)
    curation_record = gui.state.summary()
    source = "curated"
elif saved is not None:
    labels = saved["labels"]
    masks_by_group = saved.get("per_group", {})
    reference = saved.get("reference")
    config = saved.get("config", {})
    params = config.get("segmentation", {})
    merge_params = config.get("merge", {})
    curation_record = config.get("curation")
    source = f"portable: {saved['path'].name}"
elif "gui" in dir():
    labels = gui.state.merged().labels
    masks_by_group = gui.state.segment_all()
    reference = background_image(corr_by_odor)
    params = dict(gui.state.shared)
    merge_params = dict(gui.state.merge_params)
    curation_record = None
    source = "automatic"
else:
    raise FileNotFoundError("Set PORTABLE_MASK_BUNDLE or run the segmentation GUI.")

if reference is None:
    raise ValueError("The portable mask bundle has no reference image.")

if not source.startswith("portable:"):
    bundle = save_mask_bundle(
        output_path("10x_masks", ".h5"), labels,
        per_group_masks=masks_by_group,
        reference=reference,
        config={"segmentation": params, "merge": merge_params,
                "curation": curation_record},
    )
    source += f"; saved {bundle.name}"

fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(reference, cmap="gray", vmin=np.percentile(reference, 1),
          vmax=np.percentile(reference, 99.5))
ax.contour(labels > 0, levels=[0.5], colors="magenta", linewidths=0.6)
ax.set(title=f"{int(labels.max())} ROIs — {source}", xticks=[], yticks=[])

png = save_mask_overlay(output_path("masks", ".png"), reference, labels)
print(png.name)


## Trace extraction

Apply the final mask to every acquisition and write one processed HDF5 plus a PNG overlay. Temporary extraction stays on local scratch to reduce server writes and support resume. Server correlation caches are removed after extraction and QC succeed.

MATLAB reverses HDF5 dimensions; restore the Python axis order when reading:

```matlab
file = "group..._processed_YYYYMMDD.h5";
labels = h5read(file, "/masks/labels").';
roi = permute(h5read(file, "/traces/roi"), [3 2 1]); % ROI × trial × frame
neuropil = permute(h5read(file, "/traces/neuropil"), [3 2 1]);
time_s = h5read(file, "/traces/time_s");
```


In [ ]:
RUN_EXTRACTION = True

from analysis.session.finalize import finalize_session, mask_hash, verify
from analysis.session.store import read_session

existing = verify(session.output_dir)
current_hash = mask_hash(labels)
already_current = existing["status"] == "ok" and existing["mask_hash"] == current_hash

round_path = None
if RUN_EXTRACTION and not already_current:
    result = finalize_session(
        session, labels,
        per_group_masks=masks_by_group,
        segmentation_params=params,
        merge_params=merge_params,
        curation=curation_record,
        images=reference,
        pre_s=PRE_S,
        post_s=POST_S,
        neuropil=True,
        scratch_dir=SCRATCH,
        detrend=DETREND,
    )
    round_path = Path(result["path"])
elif RUN_EXTRACTION and already_current:
    round_path = session.output_dir / existing["file"]

if round_path is not None:
    from analysis.session.response_qc import response_qc

    qc = response_qc(round_path)
    print(round_path.name, qc["figure"])
    for flag in qc["flags"]:
        print(f"! {flag}")


    # The processed round now contains the masks and traces, so server caches
    # are no longer needed. Keep them when extraction or QC fails for retry.
    for cache_path in (CORR_CACHE, session.output_dir / "correlation_cache",
                       session.output_dir / "zscore_cache"):
        if cache_path.exists():
            shutil.rmtree(cache_path)
            print(f"removed server cache: {cache_path.name}")
